# 05 - Single Turn Model: comparing embeddings

The purpose of this notebook is to train one FFNN per embedding (GPT-2, Nomic, Qwen3) on single-turn data, select the best by validation PR-AUC, then check whether that model transfers to multi-turn conversations it never saw.  

#### Load Dependencies and set directory structure

In [47]:
# Load dependencies and files

# Set dependencies
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    classification_report, confusion_matrix,
    average_precision_score, roc_auc_score
)
from pyprojroot import here

# Project path anchors
REPO_ROOT = here()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"
MODEL_DIR = REPO_ROOT / "data" / "models"

In [48]:
SEED = 1234
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("device:", device)

device: mps


#### Load data and create helper functions

In [49]:
# Load the data
all_data = pd.read_parquet(PROCESSED_DATA_DIR / "all_data.parquet")

# match the values
train_mask = (all_data["split"] == "train").values
val_mask = (all_data["split"] == "val").values
test_mask = (all_data["split"] == "test").values

EMBEDDING_FILES = {
    "gpt2": "all_data_emb_gpt2.npy",
    "nomic": "all_data_emb_nomic.npy",
    "qwen3": "all_data_emb_qwen3.npy",
}

singleturn_mask = (all_data["dataset"] == "singleturn").values
multiturn_mask  = (all_data["dataset"] == "multiturn").values

def load_split(embedding, type_mask, label_col="harm"):
    embeddings = np.load(PROCESSED_DATA_DIR / EMBEDDING_FILES[embedding])
    out = {}
    for name, split_mask in [("train", train_mask), ("val", val_mask), ("test", test_mask)]:
        m = split_mask & type_mask
        out[f"X_{name}"] = embeddings[m]
        out[f"y_{name}"] = all_data.loc[m, label_col].values
    return out

def evaluate(model, loader):
    model.eval()
    all_probs, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            all_probs.append(torch.sigmoid(model(xb.to(device))).cpu())
            all_y.append(yb)
    return torch.cat(all_probs).numpy(), torch.cat(all_y).numpy()

#### Setup model architecture and load data

In [50]:
def to_loader(X, y, batch_size=64, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y.astype(np.float32), dtype=torch.float32)

    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

class FFNN(nn.Module):
    def __init__(self, in_dim, hidden_dims=(256,64), dropout=.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x).squeeze(-1)

#### Setup training loop with early stopping

In [51]:
def train_ffnn(splits, n_epochs=50, patience=5, lr=1e-3):
    train_loader = to_loader(splits["X_train"], splits["y_train"], shuffle=True)
    val_loader   = to_loader(splits["X_val"], splits["y_val"])

    model = FFNN(splits["X_train"].shape[1]).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_pr_auc, best_state, stale = -1.0, None, 0
    for epoch in range(1, n_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        val_probs, val_y = evaluate(model, val_loader)
        val_pr_auc = average_precision_score(val_y, val_probs)

        if val_pr_auc > best_pr_auc:
            best_pr_auc, best_state, stale = val_pr_auc, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            stale += 1
            if stale >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_pr_auc

#### Train one FFNN per embedding on the single turn data

In [52]:
results = {}
for name in ["gpt2", "nomic", "qwen3"]:
    splits = load_split(name, singleturn_mask)
    model, val_pr_auc = train_ffnn(splits)
    results[name] = {"model": model, "splits": splits, "val_pr_auc": val_pr_auc}
    print(f"{name:>6}  val_pr_auc={val_pr_auc:.3f}")


  gpt2  val_pr_auc=0.874
 nomic  val_pr_auc=0.897
 qwen3  val_pr_auc=0.882
